# 🌳 Session 2: Random Forest for Wheat Yield Prediction
## Practical ML for Agriculture

**Duration:** 2 hours  
**Prerequisite:** Session 1 completed  
**Goal:** Understand and implement Random Forest; compare against linear regression

---
### Session Roadmap
| Time | Topic |
|------|-------|
| 0:00–0:20 | Part 1: From decision trees to Random Forest — the concept |
| 0:20–0:50 | Part 2: Training a Random Forest model |
| 0:50–1:20 | Part 3: Feature importance & hyperparameter tuning |
| 1:20–1:50 | Part 4: Evaluation, limitations, spatial cross-validation |
| 1:50–2:00 | Wrap-up |
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor, plot_tree
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid')

print('✅ Libraries loaded!')

In [ ]:
# ── Load and prepare data (same as Session 1) ────────────────────
DATA_PATH = 'subset_WA_allcells_1989-2021.parquet'
try:
    df = pd.read_parquet(DATA_PATH)
except:
    df = pd.read_csv(DATA_PATH.replace('.parquet', '.csv'))

df_valid = df[
    (df['no_sow'] == 0) &
    (df['wheat_yield'].notna()) &
    (df['wheat_yield'] > 0)
].copy()

FEATURE_COLS = [
    'pawc_0_30_mm', 'pawc_0_60_mm', 'ph_0_30', 'profile_depth_cm',
    'rain_sum_W1_estab', 'rain_sum_W2_veg', 'rain_sum_W3_preAnth',
    'rain_sum_W4_grainFill', 'rain_sum_crop',
    'tmean_W3_preAnth', 'tmean_W4_grainFill', 'tmax_max_W4_grainFill',
    'heat_days_W3_preAnth', 'heat_days_W4_grainFill',
    'frost_days_W1_estab', 'frost_days_W2_veg',
    'vpd_mean_W3_preAnth', 'vpd_mean_W4_grainFill',
    'sowing_doy',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_valid.columns]
TARGET_COL   = 'wheat_yield'

data_model = df_valid[FEATURE_COLS + [TARGET_COL, 'year', 'lat', 'lon']].dropna()
X = data_model[FEATURE_COLS].values
y = data_model[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Data ready: {X.shape[0]:,} samples, {X.shape[1]} features')
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

---
## 🌱 Part 1: Decision Trees → Random Forest

### 1.1 What is a Decision Tree?

A **Decision Tree** splits data into groups based on simple yes/no questions about features:
```
Is rain_sum_crop > 250 mm?
  YES → Is tmean_W4 > 22°C?
          YES → predict yield = 1.2 t/ha  (hot & wet — poor outcome)
          NO  → predict yield = 3.1 t/ha  (mild & wet — good!)
  NO  → Is frost_days_W1 > 5?
          YES → predict yield = 0.8 t/ha  (dry + frosted)
          NO  → predict yield = 1.9 t/ha  (dry, no frost)
```

### ❓ Question 1.1 — Why are Single Trees Problematic?
What do you think happens if we let a tree grow very deep (many splits)?  
And if we keep it very shallow (only 2–3 splits)?

In [ ]:
# ── 1.2 Demonstrate overfitting of a single tree ──────────────────
depths = [2, 5, 10, 20, None]  # None = unlimited
results = []

for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    r2_train = r2_score(y_train, tree.predict(X_train))
    r2_test  = r2_score(y_test,  tree.predict(X_test))
    results.append({'depth': str(depth), 'train_r2': r2_train, 'test_r2': r2_test})

res_df = pd.DataFrame(results)
print('Decision Tree performance by depth:')
print(res_df.round(3).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(res_df))
ax.plot(x_pos, res_df['train_r2'], 'o-', label='Training R²', color='steelblue')
ax.plot(x_pos, res_df['test_r2'],  's--', label='Test R²',     color='tomato')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'depth={d}' for d in res_df['depth']])
ax.set_ylabel('R²')
ax.set_title('Decision Tree: Training vs Test R² by Tree Depth')
ax.legend()
ax.axhline(0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

### ❓ Question 1.2 — The Bias-Variance Trade-off
From the plot above:
1. What happens to Training R² as depth increases?
2. What happens to Test R² as depth increases past 10?
3. The gap between Train R² and Test R² is called the **generalisation gap**. When is it largest?
4. This is called the **bias-variance trade-off**. Can you explain in your own words?

In [ ]:
# ── 1.3 Visualise a shallow decision tree ─────────────────────────
small_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
small_tree.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    small_tree,
    feature_names=FEATURE_COLS,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
ax.set_title('Decision Tree (max_depth=3) — Wheat Yield Prediction', fontsize=14)
plt.tight_layout()
plt.show()
print('Interpret: each node shows the split condition and predicted yield (value) if you stop there.')

### 1.4 From One Tree to a Forest 🌲🌲🌲

**Random Forest** solves the overfitting problem by training **many trees** and averaging their predictions:

1. **Bagging:** Each tree is trained on a **random bootstrap sample** (sampling with replacement) of the training data — so each tree sees slightly different data.
2. **Feature randomness:** At each split, only a **random subset** of features is considered — so trees are decorrelated.
3. **Averaging:** Final prediction = **average** of all tree predictions.

💡 *Averaging many slightly different ("diverse") weak models reduces variance without increasing bias — this is called ensemble learning.*

---
## 🌲 Part 2: Training a Random Forest

In [ ]:
# ── 2.1 Train a Random Forest (default settings) ──────────────────
rf_model = RandomForestRegressor(
    n_estimators=100,    # number of trees
    random_state=42,
    n_jobs=-1            # use all CPU cores
)
rf_model.fit(X_train, y_train)
print('✅ Random Forest trained!')
print(f'   Trees: {rf_model.n_estimators}')
print(f'   Features per split: {rf_model.max_features}')

In [ ]:
# ── 2.2 Evaluate and compare with linear regression ───────────────
def evaluate(y_true, y_pred, label=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return {'label': label, 'rmse': round(rmse,3), 'mae': round(mae,3), 'r2': round(r2,3)}

# Linear regression baseline
scaler = StandardScaler()
lr = LinearRegression()
lr.fit(scaler.fit_transform(X_train), y_train)

rows = [
    evaluate(y_train, lr.predict(scaler.transform(X_train)), 'LR — Train'),
    evaluate(y_test,  lr.predict(scaler.transform(X_test)),  'LR — Test'),
    evaluate(y_train, rf_model.predict(X_train),             'RF — Train'),
    evaluate(y_test,  rf_model.predict(X_test),              'RF — Test'),
]
results_df = pd.DataFrame(rows)
print('=== Model Comparison ===')
print(results_df.to_string(index=False))

In [ ]:
# ── 2.3 Predicted vs Actual — Random Forest ──────────────────────
y_pred_rf_test  = rf_model.predict(X_test)
y_pred_rf_train = rf_model.predict(X_train)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (yt, yp, label) in zip(axes, [
    (y_train, y_pred_rf_train, 'Training'),
    (y_test,  y_pred_rf_test,  'Test'),
]):
    ax.scatter(yt, yp, alpha=0.15, s=5, color='seagreen')
    lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
    ax.plot(lims, lims, 'r--', linewidth=2, label='1:1 line')
    ax.set_xlabel('Actual Yield (t/ha)')
    ax.set_ylabel('Predicted Yield (t/ha)')
    ax.set_title(f'RF {label}  (R²={r2_score(yt, yp):.3f})')
    ax.legend()

plt.suptitle('Random Forest: Predicted vs Actual Wheat Yield', fontsize=14)
plt.tight_layout()
plt.show()

### ❓ Question 2.1 — RF vs Linear Regression
Compare the test R² of Random Forest vs Linear Regression:
1. Which model performs better? By how much?
2. Notice that RF train R² ≈ 0.97 but test R² is lower — why? Is this a problem?
3. Look at the scatter plots: does RF capture extreme yield values better than linear regression?

In [ ]:
# ── 2.4 How many trees do we need? ────────────────────────────────
n_trees_range = [10, 25, 50, 100, 200, 500]
train_r2s, test_r2s = [], []

for n in n_trees_range:
    rf = RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    train_r2s.append(r2_score(y_train, rf.predict(X_train)))
    test_r2s.append(r2_score(y_test,  rf.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_trees_range, train_r2s, 'o-', label='Train R²', color='steelblue')
ax.plot(n_trees_range, test_r2s,  's--', label='Test R²',  color='tomato')
ax.set_xlabel('Number of Trees')
ax.set_ylabel('R²')
ax.set_title('Random Forest: Effect of Number of Trees')
ax.legend()
plt.tight_layout()
plt.show()
print('💡 After ~100 trees, adding more trees gives diminishing returns.')

---
## 🔍 Part 3: Feature Importance

In [ ]:
# ── 3.1 Built-in feature importance (impurity-based) ─────────────
importance_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print('Feature importances (impurity-based):')
print(importance_df.round(4).to_string(index=False))

In [ ]:
# ── 3.2 Plot feature importance ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))
sorted_imp = importance_df.sort_values('importance')
ax.barh(sorted_imp['feature'], sorted_imp['importance'], color='seagreen')
ax.set_xlabel('Feature Importance (mean decrease in impurity)')
ax.set_title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

### ❓ Question 3.1 — Feature Importance
1. Which feature ranks highest? Is this consistent with the correlation analysis from Session 1?
2. Compare these importances with the linear regression coefficients. Are the rankings similar?
3. **Caution:** Impurity-based importance can be biased toward high-cardinality (many unique values) features. What alternative exists? (hint: permutation importance)

In [ ]:
# ── 3.3 Permutation importance (more reliable) ────────────────────
from sklearn.inspection import permutation_importance

perm_imp = permutation_importance(
    rf_model, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'perm_importance': perm_imp.importances_mean,
    'perm_std': perm_imp.importances_std,
}).sort_values('perm_importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, df_plot, title in zip(axes,
    [importance_df.sort_values('importance'), perm_df.sort_values('perm_importance')],
    ['Impurity-based Importance', 'Permutation Importance (test set)']):
    if 'perm_importance' in df_plot.columns:
        ax.barh(df_plot['feature'], df_plot['perm_importance'],
                xerr=df_plot['perm_std'], color='coral')
    else:
        ax.barh(df_plot['feature'], df_plot['importance'], color='seagreen')
    ax.set_xlabel('Importance')
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Hyperparameter tuning ─────────────────────────────────────
# Key hyperparameters in Random Forest:
#   n_estimators  : number of trees (more = better, but slower)
#   max_depth     : maximum depth of each tree (None = unlimited)
#   min_samples_leaf: minimum samples in a leaf (higher = smoother predictions)
#   max_features  : fraction of features to consider at each split

from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 5, 10],
    'max_features': [0.5, 0.7, 1.0],
}

# Use a small subset for speed during the session
idx_sample = np.random.RandomState(42).choice(len(X_train), min(5000, len(X_train)), replace=False)

gs = GridSearchCV(
    RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)
gs.fit(X_train[idx_sample], y_train[idx_sample])

print('Best hyperparameters:', gs.best_params_)
print(f'Best CV R²: {gs.best_score_:.3f}')

In [ ]:
# ── 3.5 Train best model on full training data ─────────────────────
best_params = gs.best_params_
best_params['n_estimators'] = 200

rf_best = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y_train)

y_pred_best = rf_best.predict(X_test)
print(f'Tuned RF — Test R²: {r2_score(y_test, y_pred_best):.3f}')
print(f'Tuned RF — Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best)):.3f} t/ha')

---
## 📏 Part 4: Advanced Evaluation

In [ ]:
# ── 4.1 Temporal cross-validation ─────────────────────────────────
# Simulate real forecasting: train on past years, test on future years
# This is more realistic than random split!

data_cv = df_valid[FEATURE_COLS + [TARGET_COL, 'year']].dropna()
years   = sorted(data_cv['year'].unique())
split_year = 2010

train_mask = data_cv['year'] <= split_year
test_mask  = data_cv['year'] >  split_year

X_tr = data_cv.loc[train_mask, FEATURE_COLS].values
y_tr = data_cv.loc[train_mask, TARGET_COL].values
X_te = data_cv.loc[test_mask,  FEATURE_COLS].values
y_te = data_cv.loc[test_mask,  TARGET_COL].values

rf_temporal = RandomForestRegressor(n_estimators=200, **{k:v for k,v in best_params.items() if k != 'n_estimators'}, random_state=42, n_jobs=-1)
rf_temporal.fit(X_tr, y_tr)

r2_temporal = r2_score(y_te, rf_temporal.predict(X_te))
r2_random   = r2_score(y_test, rf_best.predict(X_test))

print('Comparison of validation strategies:')
print(f'  Random 80/20 split         → Test R² = {r2_random:.3f}')
print(f'  Temporal split (≤2010/2011+) → Test R² = {r2_temporal:.3f}')
print()
print('💡 Temporal validation is more realistic for forecasting future seasons!')

In [ ]:
# ── 4.2 Prediction error by year ──────────────────────────────────
# See if the model struggles in specific years (e.g. extreme drought/heat)

test_subset = data_cv[test_mask].copy()
test_subset['predicted'] = rf_temporal.predict(X_te)
test_subset['residual']  = test_subset[TARGET_COL] - test_subset['predicted']

yr_rmse = test_subset.groupby('year').apply(
    lambda g: np.sqrt(mean_squared_error(g[TARGET_COL], g['predicted']))
)
yr_mean = test_subset.groupby('year')[TARGET_COL].mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].bar(yr_rmse.index, yr_rmse.values, color='coral')
axes[0].set_ylabel('RMSE (t/ha)')
axes[0].set_title('Test RMSE by Year (trained on ≤2010)')

axes[1].plot(yr_mean.index, yr_mean.values, 'o-', color='steelblue', label='Actual mean yield')
yr_pred_mean = test_subset.groupby('year')['predicted'].mean()
axes[1].plot(yr_pred_mean.index, yr_pred_mean.values, 's--', color='seagreen', label='Predicted mean yield')
axes[1].set_ylabel('Mean Yield (t/ha)')
axes[1].set_title('Actual vs Predicted Mean Yield by Year')
axes[1].legend()

plt.tight_layout()
plt.show()

### ❓ Question 4.1 — Temporal Performance
1. Are there specific years where RMSE is much higher? What might have happened those years?
2. Does the model tend to over- or under-predict in recent years? What could cause this?
3. What does it mean for farmers and policymakers if the model fails in the most extreme years?

In [ ]:
# ── 4.3 Partial dependence: how does yield respond to rainfall? ───
from sklearn.inspection import PartialDependenceDisplay

rain_idx = FEATURE_COLS.index('rain_sum_crop') if 'rain_sum_crop' in FEATURE_COLS else 0
heat_idx = FEATURE_COLS.index('heat_days_W4_grainFill') if 'heat_days_W4_grainFill' in FEATURE_COLS else 1

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
PartialDependenceDisplay.from_estimator(
    rf_best, X_test, features=[rain_idx, heat_idx],
    feature_names=FEATURE_COLS, ax=ax
)
ax[0].set_title('Partial Dependence: Seasonal Rainfall')
ax[1].set_title('Partial Dependence: Heat Days (grain fill)')
plt.suptitle('How yield changes as each feature varies (others held constant)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary Scorecard ─────────────────────────────────────────────
r2_rf  = r2_score(y_test, rf_best.predict(X_test))
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_best.predict(X_test)))
scaler2 = StandardScaler()
lr2 = LinearRegression().fit(scaler2.fit_transform(X_train), y_train)
r2_lr  = r2_score(y_test, lr2.predict(scaler2.transform(X_test)))
rmse_lr = np.sqrt(mean_squared_error(y_test, lr2.predict(scaler2.transform(X_test))))

print('=' * 55)
print('  RANDOM FOREST — SESSION 2 SUMMARY')
print('=' * 55)
print(f'  Linear Regression Test R²  : {r2_lr:.3f}  RMSE: {rmse_lr:.3f} t/ha')
print(f'  Random Forest Test R²      : {r2_rf:.3f}  RMSE: {rmse_rf:.3f} t/ha')
print(f'  R² improvement             : +{r2_rf - r2_lr:.3f}')
print()
print('  STRENGTHS')
print('    ✅ Handles non-linear relationships')
print('    ✅ Captures feature interactions automatically')
print('    ✅ Robust to outliers')
print('    ✅ Built-in feature importance')
print('    ✅ No need for feature scaling')
print()
print('  LIMITATIONS')
print('    ❌ Black box — harder to interpret than linear regression')
print('    ❌ Slow to train on very large datasets')
print('    ❌ Cannot extrapolate beyond training data range')
print('    ❌ Memory-intensive')
print('=' * 55)
print('\nNext session: XGBoost — gradient boosting for even better performance!')

---
## 📝 Session 2 — Take-Home Exercises

**Exercise A:** Add more features (monthly rainfall, stress indices) and retrain the Random Forest. How does adding features change performance and feature importances?

**Exercise B:** Try `min_samples_leaf=1` vs `min_samples_leaf=20`. Which reduces overfitting more? Plot the training and test R².

**Exercise C:** Use `out-of-bag (OOB)` score instead of a validation split:
```python
rf = RandomForestRegressor(n_estimators=100, oob_score=True)
rf.fit(X_train, y_train)
print(rf.oob_score_)  # free estimate using the bags not seen by each tree
```
How does OOB score compare to test set R²?

**Exercise D:** Spatial cross-validation — split by geographic region (e.g., north vs south WA based on latitude). Does the model trained on one region generalise to another?

---
*Session 3 will cover **XGBoost** — a gradient boosting approach that often outperforms Random Forest.*